In [ ]:
#find all direct children of FORM_SECTION across a set of documents
query_element = "FORM_SECTION"
list_of_descendants = []

for i, file in enumerate(tqdm(xml_files_list)):
    
    tree = ET.parse(xml_files_list[i])  #initialize the tree
    root = tree.getroot()   #get root
    
    #retrieve the root tag and strip the xmlns attribute value
    root_tag = str(root.tag)
    xmlns = root_tag.split("}")[0] + "}"
    
    element = str(tree.findall('.//{}{}/'.format(xmlns, query_element))[0]).split("}")[1].split("""'""")[0] #find the first direct descendant of FORM_SECTION in all xml files
    list_of_descendants.append(element)

#Find a list of unique elements that are direct descendants to the FORM_SECTION
#np.savetxt("Data/descendants_of_form_section.csv", list_of_descendants, fmt='%s')

len(set(list_of_descendants))

In [1]:
#THIS CELLBLOCK CONTAINS CODE WHICH IS CURRENLTY NOT USEFULL BUT MIGHT BE REQUIRED IN FUTURE WORK. DO NOT REMOVE UNLESS CERTAIN

def count_tags(filename):
        """This function takes a file as input, creates a data_dict with xml tags as keys, and counts of unique tags as values. It returns the maximum value"""
        
        my_tags = []

        for event, element in ET.iterparse(filename):
            my_tags.append(element.tag)

        my_keys = Counter(my_tags).keys()
        my_values = Counter(my_tags).values()
        my_dict = dict(zip(my_keys, my_values))
        
        return my_dict

In [ ]:
df_descendants = pd.read_csv("Data/descendants_of_form_section.csv")

In [ ]:
#first inspect if we can select for the language attribute across these various descendants to only get the english info
unique_descendants = list(set(list_of_descendants))
indexes_files = []

#retrieve the index of the items to get the xml file names
for element in unique_descendants:
    indexes_files.append(list_of_descendants.index(element))

for i, file_index in enumerate(indexes_files):
    print(xml_files_list[file_index], unique_descendants[i])

In [ ]:
#find the total amount of unique elements in an english contract, count the frequency over all documents, and retrieve first file of occurence
tag_file_count = {}
query_element = "FORM_SECTION"
len_xml_file_name = 15

for i, file in enumerate(tqdm(xml_files_list)):
  #parse the tree and get the root element
  tree = ET.parse(file)
  root = tree.getroot()

  #get the root tag and split it to get the value for the xmlns attribute
  root_tag = str(root.tag)
  xmlns_value = root_tag.split("}")[0] + "}"

  for j, contract_info in enumerate(root.findall(".//{}{}//*[@LG='EN']//*".format(xmlns_value, query_element))):
    
    key = str(contract_info.tag)[len(xmlns_value):] #strip the tag to only maintain the element without xmlns_value as a unique key
    file_name = str(file)[-len_xml_file_name:]
    
    if key not in tag_file_count.keys(): #if the tag is not in the dict, add it and give it frequency 1
      value = 1
      tag_file_count[key] = [value, file_name]

    else: #if the tag is already in the dict, update the frequency by 1
      value = tag_file_count[key][0] + 1
      tag_file_count[key] = [value, file_name]

#sort data_dict, take list output and return to a data_dict
sorted_tag_file_count = dict(sorted(tag_file_count.items(), key = lambda x:x[1], reverse = True))

In [ ]:
with open("Data/Unique tags and frequencies.pickle", "wb") as f:
    pickle.dump(sorted_tag_file_count, f)

In [ ]:
def extract_data(collection: dict, xmlns_len: int, start: str, txt_tags: list, attrib_tags: list):
    """This function takes in an empty dictionary and an xml tag and fills the dictionary with data based on a set of conditions"""
    for j, xml_tag in enumerate(root.findall(start)):
        clean_tag = xml_tag.tag[xmlns_len:]
                
        if (clean_tag in txt_tags and
                clean_tag in collection):                           #text tag + existing tag
            if xml_tag.text != None:
                collection[clean_tag].append(xml_tag.text)             #text is present
            else:                                                        #text not present
                collection[clean_tag].append(None) 
        elif (clean_tag in attrib_tags and
                clean_tag in collection):                           #attribute tag + existing tag
            if xml_tag.attrib.values() != None:
                attribute_value = list(xml_tag.attrib.values())
                collection[clean_tag].append(attribute_value[0])    #atrribute is present
            else:                                                        #attribute not present
                collection[clean_tag].append(None)
        elif (clean_tag in txt_tags and
                clean_tag not in collection):                           #text tag + new tag
            if xml_tag.text != None:
                collection[clean_tag] = [xml_tag.text]                  #text is present
            else:                                                        #text not present
                collection[clean_tag] = [None]
        elif (clean_tag in attrib_tags and
                clean_tag not in collection):                           #attribute tag + new tag
            if xml_tag.attrib.values() != None:
                attribute_value = list(xml_tag.attrib.values())
                collection[clean_tag] = [attribute_value[0]]             #atrribute is present
            else:                                                        #attribute not present
                collection[clean_tag] = [None]
    return collection

In [ ]:
#initialize empty lists to be filled with column names
cols_element_tags = []
cols_attrib_tag = []

#initialize empty lists to be filled with attribute and text values
attribute_values = []
text_values = []

#initialize empty list to be filled with all attribute keys
attribute_keys = []

query_element = "FORM_SECTION"

for i, file in enumerate(tqdm(xml_files_list[:10])):
    
    #parse the tree and get the root element
    tree = ET.parse(file)
    root = tree.getroot()

    #get the root tag and split it to get the value for the xmlns attribute
    root_tag = str(root.tag)
    xmlns_value = root_tag.split("}")[0] + "}"

    #give me all descendant elements of the element which has attribute value LG=EN where the parent element is Form_section
    for j, contract_info in enumerate(root.findall(".//{}{}//*[@LG='EN']//*".format(xmlns_value, query_element))): 

        #add to the columns the name of each element for which we want to store corresponding data
        cols_element_tags.append(contract_info.tag.strip(xmlns_value))

        #make a list of column values containing tags and attributes and retrieve attribute values
        if len(contract_info.attrib.keys()) == 0:
            
            attribute_key = ["None"]
            attribute_value = ["None"]
            attribute_values.append(attribute_value[0])
        
        else:
            attribute_key = list(contract_info.attrib.keys())
            attribute_keys.append(attribute_key)

            attribute_value = list(contract_info.attrib.values())
            attribute_values.append(attribute_value[0]) #xml attribute:value pairs are stored as a list but can only
                                                         #have one value and attribute which is why indexing at [0] always works

        cols_attrib_tag.append((cols_element_tags[j] + ": " + attribute_key[0]))

        #append the text value of an element in the english contract
        text_values.append(contract_info.text)

    #for contract_info in root.findall('.//{}[@LG="EN"]//*'.format(tag)):
    #values_text.append(contract_info.text)

    #combine columns and rows in one big list
    column_names = cols_element_tags + cols_attrib_tag
    rows = text_values + attribute_values

    #create the dataframe from the two lists
    df = pd.DataFrame(data = rows, index = column_names).T

In [ ]:
#find those elements for which the child element has tag <p> 
query_element = "CPV_CODE"

for i, file in enumerate(tqdm(xml_files_list[0:1500])):
  #parse the tree and get the root element
  tree = ET.parse(file)
  root = tree.getroot()
  
  #get the root tag and split it to get the value for the xmlns attribute
  root_tag = str(root.tag)
  xmlns_value = root_tag.split("}")[0] + "}"

  for j, contract_info in enumerate(root.findall(".//*[@LG='EN']//{}{}".format(xmlns_value, query_element))):
        print(contract_info.tag, contract_info.text, contract_info.attrib)

In [ ]:
#print all attributes, values and texts of x elements in data_dict
query_element = "CONTENTS"

for i, file in enumerate(tqdm(xml_files_list[:1000])):
    #parse the tree and get the root element
    tree = ET.parse(file)
    root = tree.getroot()
    
    #get the root tag and split it to get the value for the xmlns attribute
    root_tag = str(root.tag)
    xmlns_value = root_tag.split("}")[0] + "}"

    for j, contract_info in enumerate(root.findall(".//*[@LG='EN']//{}{}//{}{}".format(xmlns_value, query_element, xmlns_value, "P"))):
            print(contract_info.tag, contract_info.attrib, contract_info.text)

In [ ]:
#open the tar.gz file as a tar archive
data_2016 = r"C:\Users\bbinnend\Desktop\D&E\Thesis\Data\2016-01-01.2016-31-01\2016-01.tar.gz"
data_2021 = r"C:\Users\bbinnend\Desktop\D&E\Thesis\Data\2021-01.tar.gz"

tar_archive = tarfile.open(name=r"C:\Users\bbinnend\Desktop\D&E\Thesis\Data\2016-01.tar.gz", mode="r:gz")

#extract the tar archive in a directory of choice
#tar_archive.extractall("data/2021-01-01.2021-31-01/extracted.tar.gz.01")

#Initialize an empty list and fill it with path to each .xml file
xml_files_list=[]

for file in tar_archive.getnames():
    if file[-4:] == ".xml":
        xml_files_list.append(str("Data/2016-01-01.2016-31-01/extracted.tar.gz.01/" + file))

tar_archive.close()

In [ ]:
tags = ["CPV_CODE", "COUNTRY", "CODE", "TI_DOC", "CONTENTS/P", "OBJ_NOT/P", "AWARD_OF_CONTRACT/DAY",
"AWARD_OF_CONTRACT/MONTH", "AWARD_OF_CONTRACT/YEAR", "OFFERS_RECEIVED_NUMBER", "VALUE_COST", "CONTRACT_TITLE/P", "TITLE_CONTRACT", 
"FOR_READ/P", "ORDER_C", "CRITERIA", "WEIGHTING", "NOTICE_DISPATCH_DATE/DAY", 
"NOTICE_DISPATCH_DATE/MONTH", "NOTICE_DISPATCH_DATE/YEAR", "TYPE_CONTRACT", "TYPE_OF_ACTIVITY", "SHORT_CONTRACT_DESCRIPTION", 
"TYPE_OF_CONTRACTING_AUTHORITY", "NATURE_QUANTITY_SCOPE/P", "HIGH_VALUE", "LOW_VALUE"]
excluded = ["CONTENTS", ]

substitutable_groups = {"CPV_CODE": ["CPV_ADDITIONAL", "CODE", "CPV_MAIN", "CPV"], 
                  "AWARD_OF_CONTRACT": ["CONTRACT_AWARD_DATE"],
                  "CONTRACT_TITLE/P": ["TITLE_CONTRACT/P"],
                  "COUNTRY": ["NUTS"], 
                  "NATURE_QUANTITY_SCOPE/P": ["TOTAL_QUANTITY_OR_SCOPE/P", "QUANTITY_SCOPE/P"],
                  "TYPE_CONTRACT": ["FD_CONTRACT"]
                }
complementary_groups = {"contract_description": ["TI_DOC", "NATURE_QUANTITY_SCOPE"]}

attribute_tags = ["COUNTRY", "CPV_CODE", "VALUE_COST"]
text_tags = ["OFFERS_RECEIVED_NUMBER", "CONTRACT_TITLE"]
hierarchy_tags = ["CRITERIA_DEFINITION", "CONTRACT_AWARD_DATE", "TI_DOC"]

In [ ]:
#lists of elements divided across its themes
notice_sections = ["CONTRACTING_BODY", "OBJECT_CONTRACT", "OBJECT_DESCRIPTION", "LEFTI", "PROCEDURE", "COMPLEMENTARY_INFO"]

contracting_body_required = ["OFFICIALNAME"]
contracting_body_optional = ["CA_TYPE"]

object_contract_required = ["TITLE", "CPV_MAIN", "CPV_CODE", "TYPE_CONTRACT"]
object_contract_optional = ["SHORT_DESCR", "VAL_ESTIMATED_TOTAL", "OBJECT_DESCR"]

object_description_required = ["CPV_ADDITIONAL", "CPV_CODE", "AC", ]
object_description_optional = ["TITLE", "SHORT_DESCR", "VAL_OBJECT", "DURATION", "DATE_START", "DATE_END", "RENEWAL", "NO_RENEWAL"]

procedure_required = ["PT_OPEN", "ACCELERATED_PROC", "PT_RESTRICTED", "PT_COMPETITIVE_NEGOTIATION", "PT_COMPETITIVE_DIALOGUE", "PT_INNOVATION_PARTNERSHIP"]

complementary_information = ["RECURRENT_PROCUREMENT", "NO_RECURRENT_PROCUREMENT"]

In [ ]:
#lists of elements divided across its themes
notice_sections = ["CONTRACTING_BODY", "OBJECT_CONTRACT", "OBJECT_DESCR", "LEFTI", "PROCEDURE", "COMPLEMENTARY_INFO"]

contracting_body= {"attribute_tags": [], "text_tags": ["OFFICIALNAME"], "attribute_tags": ["CA_TYPE"]}

object_contract = {"attribute_tags": ["CPV_CODE", "TYPE_CONTRACT"], "text_tags": [], "hierarchy_text_tags": ["TITLE"]}
["SHORT_DESCR", "VAL_ESTIMATED_TOTAL", "OBJECT_DESCR"]

#EVERYTHING ABOVE THIS HAS ALREADY BEEN ADDED TO THE XPATH LISTS

object_description = {"attribute_tags": [], "text_tags": [], "hierarchy_text_tags": []}
["CPV_ADDITIONAL", "CPV_CODE", "AC", "SHORT_DESCR", "VAL_OBJECT", "DURATION", "DATE_START", "DATE_END", "RENEWAL", "NO_RENEWAL"]

procedure = {"attribute_tags": [], "text_tags": [], "hierarchy_tags": []}
["PT_OPEN", "ACCELERATED_PROC", "PT_RESTRICTED", "PT_COMPETITIVE_NEGOTIATION", "PT_COMPETITIVE_DIALOGUE", "PT_INNOVATION_PARTNERSHIP"]

complementary_information = {"attribute_tags": [], "text_tags": [], "hierarchy_tags": []}
["RECURRENT_PROCUREMENT", "NO_RECURRENT_PROCUREMENT"]

In [ ]:
xpath_text = [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY//{}OFFICIALNAME".format(xmlns_value, xmlns_value, xmlns_value),
            ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}SHORT_DESCR//{}P".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value),
            ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}TITLE//{}P".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value),
            ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR/{}TITLE//{}P".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value),
            ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}DATE_START".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value),
            ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}DATE_END".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value),
            ]

xpath_attribute = [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY//{}CA_TYPE".format(xmlns_value, xmlns_value, xmlns_value),
                   ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT//{}CPV_MAIN//{}CPV_CODE".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value),
                   ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT//TYPE_CONTRACT".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value)]

xpath_text_attribute = [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}VAL_ESTIMATED_TOTAL".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value),
                        ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}DURATION".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value),
                        ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}VAL_OBJECT".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value),
                        ]

xpath_attribute_stack = [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR/{}CPV_ADDITIONAL//{}CPV_CODE".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value),
                         ]

xpath_text_stack = [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}AC//*".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value)]

xpath_boolean = [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}NO_RENEWAL".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value),
                 ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}RENEWAL".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value),
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}PT_OPEN".format(xmlns_value, xmlns_value, xmlns_value),
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}ACCELERATED_PROC".format(xmlns_value, xmlns_value, xmlns_value),
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}PT_RESTRICTED".format(xmlns_value, xmlns_value, xmlns_value),
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}PT_COMPETITIVE_NEGOTIATION".format(xmlns_value, xmlns_value, xmlns_value),
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}PT_COMPETITIVE_DIALOGUE".format(xmlns_value, xmlns_value, xmlns_value),
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}PT_COMPETITIVE_NEGOTIATION".format(xmlns_value, xmlns_value, xmlns_value),
                 ".//{}FORM_SECTION//*[@LG='EN']//{}COMPLEMENTARY_INFO/{}RECURRENT_PROCUREMENT".format(xmlns_value, xmlns_value, xmlns_value),
                 ".//{}FORM_SECTION//*[@LG='EN']//{}COMPLEMENTARY_INFO/{}NO_RECURRENT_PROCUREMENT".format(xmlns_value, xmlns_value, xmlns_value)
                ]

xpath_expression = ".//{}FORM_SECTION//*[@LG='EN']//{}COMPLEMENTARY_INFO/{}INFO_ADD".format(xmlns_value, xmlns_value, xmlns_value)

#find those elements for which the child element has tag <p> 

for i, file in enumerate((xml_files_list[:1000])):
  #parse the tree and get the root element
  tree = ET.parse(file)
  root = tree.getroot()
  
  #get the root tag and split it to get the value for the xmlns attribute
  root_tag = str(root.tag)

  for j, contract_info in enumerate(root.findall(xpath_expression)):
        print(contract_info.tag, contract_info.text, contract_info.attrib, i, file)

In [ ]:
def standardize_collection(collection: dict, txt_tags: list, attrib_tags: list):
    """This function takes in a collection of data on an xml file and checks 
        if there are key:value pairs missing for standardization purposes"""
    
    for key in txt_tags:
        if key not in collection:
            collection[key] = None
        else:
            continue
        
    for key in attrib_tags:
        if key not in collection:
            collection[key] = None
        else:
            continue
        
    return collection

In [ ]:
def extract_data(collection: dict, xmlns_len: int, start: str, txt_tags: list, attrib_tags: list):
    """This function takes in an empty dictionary and an xml tag and fills the dictionary with data based on a set of conditions"""
    for j, xml_tag in enumerate(root.findall(start)):
        clean_tag = xml_tag.tag[xmlns_len:]

        if (clean_tag in txt_tags and
                clean_tag not in collection):                           #text tag + new tag
            if xml_tag.text != None:
                collection[clean_tag] = xml_tag.text                  #text is present
            else:                                                        #text not present
                collection[clean_tag] = None
        elif (clean_tag in attrib_tags and
                clean_tag not in collection):                           #attribute tag + new tag
            if xml_tag.attrib.values() != None:
                attribute_value = list(xml_tag.attrib.values())
                collection[clean_tag] = attribute_value[0]             #atrribute is present
            else:                                                        #attribute not present
                collection[clean_tag] = None
        else:
            continue
    
    collection = standardize_collection(collection = collection, txt_tags = txt_tags, attrib_tags = attrib_tags)

    return collection

-----
RESEARCH
-----
-----

-----THE CODE BELOW CALCULATED THE AMOUNT OF TIMES THE CONTRACTING AUTHORITY IS CLASSIFIED AS "OTHER". THIS IS ABOUT 21% WHICH IS A LOT SO IT MUST BE INTEGRATED INTO THE MAIN CODE TO ACCOUNT FOR IT.----- THIS IS DONE

In [ ]:
#figure out if there is no CA_TYPE, how often is the CA_TYPE_OTHER present?
query = ".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}CA_TYPE_OTHER".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value)
test_cases = []
results = []

for index in data_dict:
    if data_dict[index]["ca_type"] == None:
        test_cases.append(prefiltered_docs[index])

for file in test_cases:

    #parse the tree and get the root element
    tree = ET.parse(file)
    root = tree.getroot()
    
    if len(root.findall(query)) > 0:
        results.append(file)

print("The amount of test cases where no CA_TYPE was found is: ", len(test_cases), "\n", 
        "The amount of cases where CA_TYPE_OTHER was found when CA_TYPE was absent is: ", len(results), "\n",
        "Equaling a percentage of: ", str(len(results) / len(test_cases) * 100))

-----THE CODE BELOW CALCULATED THE AMOUNT OF TIMES THE CONTRACTING AUTHORITY ACTIVITY IS CLASSIFIED AS "OTHER". THIS IS ABOUT 40% WHICH IS A LOT SO IT MUST BE INTEGRATED INTO THE MAIN CODE TO ACCOUNT FOR IT.-----

In [ ]:
#figure out if there is no CA_TYPE, how often is the CA_TYPE_OTHER present?
query = ".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}CA_ACTIVITY_OTHER".format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value)
test_cases = []
results = []

for index in data_dict:
    if data_dict[index]["ca_activity"] == None:
        test_cases.append(prefiltered_docs[index])

for file in test_cases:

    #parse the tree and get the root element
    tree = ET.parse(file)
    root = tree.getroot()
    
    if len(root.findall(query)) > 0:
        results.append(file)

print("The amount of test cases where no CA_ACTIVITY was found is: ", len(test_cases), "\n", 
        "The amount of cases where CA_ACTIVITY_OTHER was found when CA_ACTIVITY was absent is: ", len(results), "\n",
        "Equaling a percentage of: ", str(len(results) / len(test_cases) * 100))

----- IT LOOKS LIKE <br> 
                        .//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT//{}VAL_TOTAL AND 
                    <br>
                        .//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}VALUES/{}VAL_ESTIMATED_TOTAL
                    <br>
yield similar results for when the tags occur both in the same document. This gives rise to the idea that if only one of them is present, we can use them interechangeably to determine the value of the contract, enhancing the quality of the dataset. -----
<br> they are the same in 82.5% of the time

In [ ]:
#print all attributes, values and texts of x elements in data_dict
queries = [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT//{}VAL_TOTAL",
           ".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}VALUES/{}VAL_ESTIMATED_TOTAL"]

comparing_value = {}

for i, file in enumerate(xml_files_list[:10000]):
    #parse the tree and get the root element
    tree = ET.parse(file)
    root = tree.getroot()
    temp_dict = {"VAL_TOTAL": None, "VAL_ESTIMATED_TOTAL": None}
    
    #get the root tag and split it to get the value for the xmlns attribute
    for query in queries:
        query = query.format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value)
        key = query.split("}")[-1]

        for k, contract_info in enumerate(root.findall(query)):
            if contract_info == None:
                temp_dict[key] = None
            else:
                temp_dict[key] = contract_info.text

    if (temp_dict["VAL_TOTAL"] != None and 
        temp_dict["VAL_ESTIMATED_TOTAL"] != None):
        comparing_value[i] = temp_dict

In [ ]:
#find the total amount of unique elements in an english contract, count the frequency over all documents, and retrieve first file of occurence
tag_file_count = {}
query_element = "FORM_SECTION"
len_xml_file_name = 15

for i, file in enumerate(tqdm(xml_files_list)):
  #parse the tree and get the root element
  tree = ET.parse(file)
  root = tree.getroot()

  #get the root tag and split it to get the value for the xmlns attribute
  root_tag = str(root.tag)
  xmlns_value = root_tag.split("}")[0] + "}"

  for j, contract_info in enumerate(root.findall(".//{}{}//*[@LG='EN']//*".format(xmlns_value, query_element))):
    
    key = str(contract_info.tag)[len(xmlns_value):] #strip the tag to only maintain the element without xmlns_value as a unique key
    file_name = str(file)[-len_xml_file_name:]
    
    if key not in tag_file_count.keys(): #if the tag is not in the dict, add it and give it frequency 1
      value = 1
      tag_file_count[key] = [value, file_name]

    else: #if the tag is already in the dict, update the frequency by 1
      value = tag_file_count[key][0] + 1
      tag_file_count[key] = [value, file_name]

#sort data_dict, take list output and return to a data_dict
sorted_tag_file_count = dict(sorted(tag_file_count.items(), key = lambda x:x[1], reverse = True))

In [ ]:
#with open("Data/comparing_contract_values.pickle", "wb") as f:
#    pickle.dump(contract_values, f)

----CODE BELOW BELONGS TO EXPLORATION------

In [ ]:
#seems like val_estimated_total is the total estimated value of the contract and val_total is the value per lot
##get the currency columns
value_names_unfiltered = []

for column in df_data.columns:
    if "award_contract/val_total" in column:
        value_names_unfiltered.append(column)
    else:
        continue

emptiness_criteria_val_total = []

In [ ]:
delta_emptiness_criteria = []
for i, percentage in enumerate(emptiness_criteria_val_total):
if i+1 < len(emptiness_criteria_val_total):
delta_emptiness_criteria.append(emptiness_criteria_val_total[i+1] - emptiness_criteria_val_total[i])

index_max_drop_criteria = delta_emptiness_criteria.index(max(delta_emptiness_criteria))

criteria_weighting_column_names = criteria_column_names_unfiltered["criterion"] + criteria_column_names_unfiltered["weighting"]

for column in criteria_weighting_column_names:
if int(column.split(":")[1]) > index_max_drop_criteria:
df_data.drop(column, inplace = True, axis = 1)

index_max_drop_criteria

In [ ]:
#get the currency columns
currency_column_names = []

for column in df_data.columns:
    if "currency" in column:
        currency_column_names.append(column)

#The code below takes the currency from any of the currency columns, constructs a list from it, drops all currency columns and combines them into one
currency_column = []

for row in range(len(df_data)):
    
    for cell in df_data[currency_column_names].iloc[row]:
        cell_value = None
        if cell != None:
            cell_value = cell
        else:
            continue
        
    currency_column.append(cell_value)

df_data.drop(currency_column_names, axis = 1, inplace = True)
df_data["currency"] = currency_column

In [ ]:
##get the currency columns
criteria_column_names_unfiltered = {"criterion": [], "weighting": []}

for column in df_data.columns:
    if "criterion" in column:
        criteria_column_names_unfiltered["criterion"].append(column)
    elif "weighting" in column:
        criteria_column_names_unfiltered["weighting"].append(column)
    else:
        continue

#maximum drop in criteria is after 6 criteria as in a 30 percent drop compared to the previous criteria. As such, drop all columns after 6 criteria as they are all empt
emptiness_criteria = []

for column in criteria_column_names_unfiltered["criterion"]:
    percentage = df_data[column].isnull().sum() / len(df_data) * 100
    emptiness_criteria.append(percentage)

delta_emptiness_criteria = []

for i, percentage in enumerate(emptiness_criteria):
    if i+1 < len(emptiness_criteria):
        delta_emptiness_criteria.append(emptiness_criteria[i+1] - emptiness_criteria[i])

index_max_drop_criteria = delta_emptiness_criteria.index(max(delta_emptiness_criteria))

criteria_weighting_column_names = criteria_column_names_unfiltered["criterion"] + criteria_column_names_unfiltered["weighting"]

for column in criteria_weighting_column_names:
    if int(column.split(":")[1]) > index_max_drop_criteria:
        df_data.drop(column, inplace = True, axis = 1)

index_max_drop_criteria

In [ ]:
#let's try and figure out if there are similar values in the criterion columns
criterion_column_names = []

for column in df_data.columns:
    if "criterion" in column:
        criterion_column_names.append(column)

df_criterion = pd.melt(df_data[criterion_column_names])
values_criteria = df_criterion["value"].value_counts().keys().tolist()
counts_criteria = df_criterion["value"].value_counts().tolist()

df_criterion.isnull().sum() #the amount of null counts in this df is 1271, kind of puts things into perspective

#make a nice little bar plot to show the results
fig, ax = plt.subplots()

ax.bar(values_criteria[:40], counts_criteria[:40])
ax.set_ylabel("Counts")
ax.set_xlabel("Values")
plt.title("Distribution of criteria across tenders")

plt.xticks(rotation='vertical')
plt.show()

In [ ]:
#asssess the amount of columns and check how spare some columns are (116)
amount_of_columns = len(df_data.columns)
value_count_cpv_codes = df_data["cpv_code"].value_counts()
value_count_contract_type = df_data["type_contract"].value_counts()
value_count_ca_type = df_data["ca_type"].value_counts()
value_count_countries = df_data["country"].value_counts()
value_count_procedure = df_data["procedure"].value_counts()
value_count_procedure = df_data["procedure"].value_counts()
value_count_procedure = df_data["procedure"].value_counts()

print("\n\n", value_count_contract_type, value_count_ca_type, value_count_countries)

In [ ]:
#what does the distribution of the criteria weighting look like?
weighting_columns = []

for column in df_data.columns:
    if "weighting" in column:
        weighting_columns.append(column)

weight_data = df_data[weighting_columns].melt().fillna(0)["value"].tolist()
weight_data_float = []

for weight in weight_data[:50]:
    if type(weight) == str:
        if "%" in weight:
            weight_data_float.append(float((weight.split(" ")[0]).replace(",", ".")))
        else:
            weight_data_float.append(float(weight))
    elif type(weight) == int:
        weight_data_float.append(float(weight))
    else:
        continue

fig, ax = plt.subplots()
ax.boxplot(x = weight_data_float)
ax.set_title("Distribution of criteria weights")
plt.show()

In [ ]:
counter = 0
not_identical = []
for key in contract_values:
    if contract_values[key]["VAL_TOTAL"] == contract_values[key]["VAL_ESTIMATED_TOTAL"]:
        counter += 1
    else:
        not_identical.append(key)

results = counter / len(contract_values) * 100
print("When both VAL_TOTAL and VAL_ESTIMATED_TOTAL are present, they are equal {}{} of the time".format(results, "%"))

difference_value_contract = []

for index in not_identical:
    difference_value_contract.append(((float(comparing_contract_values[index]["VAL_TOTAL"]) - float(comparing_contract_values[index]["VAL_ESTIMATED_TOTAL"])) / float(comparing_contract_values[index]["VAL_ESTIMATED_TOTAL"])) * 100)

df_delta_contract_val = pd.DataFrame(difference_value_contract, columns=["difference %"])
df_delta_contract_val.describe()

#the plot really shows how on average, the payout seems to be that the contract award price is a little higher than initial price (0.86%)
#but there is a long tail on the upside, indicating that some contracts are obtained for a lot better price. conclusion: search for the awarded price,
#if it is not there, use the initial estimation from the CA
df_delta_contract_val.boxplot(column = ["difference %"])
plt.show()

In [ ]:
for i in range(0, len(indices_dif_cur)):
    if df_data_2020.iloc[i]["award_contract/val_total[@currency]"] != "EUR" and df_data_2020.iloc[i]["object_contract/val_total[@currency]"] != "EUR":
        print(i)

In [ ]:
indices_unknown_currencies = []

for key in tqdm(list(money_coordinates.keys())):
    for money_collection in money_coordinates[key]:
        index_currency_exchange_year = 0
        assign_to_column = money_collection[1]
        original_currency =  money_collection[2]
        original_values = str(money_collection[0]).split(" ")

        for currency in df_currency_exchange.columns:
            if currency == original_currency:
                total_new_amount = 0
                for original_amount in original_values:
                    original_amount = original_amount.split(".")[0] #just keep the whole number rather than what is after the comma
                    if len(re.findall("([\d.]*\d+)", original_amount)) < 1: #sometime there will be some piece of string which cannot be converted to float
                        continue
                    else:
                        new_amount = int(float(original_amount) / df_currency_exchange[currency][index_currency_exchange_year])
                        total_new_amount += new_amount
            else:
                continue

        df_data_2020.at[key, assign_to_column] = total_new_amount

In [ ]:
#first get the indices of all rows where there is an amount of money

#money_indices = list(OrderedSet([i for i in tqdm(range(0, len(df_data_2020))) for col in money_columns if df_data_2020[col][i] != None and pd.isna(df_data_2020[col][i]) != True]))

money_coordinates = {}

for i in tqdm(range(0, len(df_data_2020))):
    money_per_row = []
    for col in money_columns:
        currency_col = col.split(":")[0] + "[@currency]"
        if df_data_2020[col][i]!= None and pd.isna(df_data_2020[col][i]) != True:
            #currency_col = col.split(":")[0] + "[@currency]"
            money_per_row.append([df_data_2020[col][i], col, df_data_2020[currency_col][i]])
            money_coordinates[i] = money_per_row

In [ ]:
indices_unknown_currencies = []

for key in tqdm(list(money_coordinates.keys())):
    for money_collection in money_coordinates[key]:
        index_currency_exchange_year = 0
        assign_to_column = money_collection[1]
        original_currency =  money_collection[2]
        original_money_values = str(money_collection[0]).split(" ")

        if original_currency in df_currency_exchange.columns:
            total_new_amount = 0
            for original_amount in original_money_values:
                original_amount = original_amount.split(".")[0] #just keep the whole number rather than what is after the comma
                if len(re.findall("([\d.]*\d+)", original_amount)) < 1: #sometime there will be some piece of string which cannot be converted to float
                    print(original_amount)
                    continue
                else:
                    new_amount = int(float(original_amount) / df_currency_exchange[original_currency][index_currency_exchange_year])
                    total_new_amount += new_amount
            df_data_2020.at[key, assign_to_column] = total_new_amount
        else:
            indices_unknown_currencies.append(key)

In [ ]:
money_list_test = [{'index': 2,                  #this test case has None's, but not similar currencies, so the value is set to None
 'object_contract/val_estimated_total: 0': '714297.50',
 'object_contract/val_estimated_total[@currency]': None,
 'object_contract/val_object: 0': '714297.50',
 'object_contract/val_object[@currency]': 'GBP',
 'object_contract/val_total: 0': '714297.50',
 'object_contract/val_total[@currency]': 'EUR',
 'award_contract/val_estimated_total': '714297.50'},
 {'index': 3,                   #this test case has None's, and a foreign currency, so the value is set to None
 'object_contract/val_estimated_total: 0': '714297.50',
 'object_contract/val_estimated_total[@currency]': None,
 'object_contract/val_object: 0': '714297.50',
 'object_contract/val_object[@currency]': 'ABC',
 'object_contract/val_total: 0': '714297.50',
 'object_contract/val_total[@currency]': 'EUR',
 'award_contract/val_estimated_total': '714297.50'},
 {'index': 5,                   #This test case has only None's, so the index is remembered for dropping later on
 'object_contract/val_estimated_total: 0': '714297.50',
 'object_contract/val_estimated_total[@currency]': None,
 'object_contract/val_object: 0': '714297.50',
 'object_contract/val_object[@currency]': None,
 'object_contract/val_total: 0': '714297.50',
 'object_contract/val_total[@currency]': None,
 'award_contract/val_estimated_total': '714297.50'},
 {'index': 6,                   #This test case has only None's, so the index is remembered for dropping later on
 'object_contract/val_estimated_total: 0': '714297.50',
 'object_contract/val_estimated_total[@currency]': 'EUR',
 'object_contract/val_object: 0': '714297.50',
 'object_contract/val_object[@currency]': 'EUR',
 'object_contract/val_total: 0': '714297.50',
 'object_contract/val_total[@currency]': None,
 'award_contract/val_estimated_total': '714297.50'},
{'index': 1, #this test case has none's but similar currencies, so the None is set to EUR
 'object_contract/val_estimated_total: 0': '714297.50',
 'object_contract/val_estimated_total[@currency]': None,
 'object_contract/val_object: 0': '714297.50',
 'object_contract/val_object[@currency]': 'EUR',
 'object_contract/val_total: 0': '714297.50',
 'object_contract/val_total[@currency]': 'EUR',
 'award_contract/val_estimated_total': '714297.50'}]

In [ ]:
df_cur_none = df_data_2020.iloc[rows_no_currencies]

euro_zone_rows = []
for i, original_index in enumerate(df_cur_none.index.tolist()):
    if df_cur_none['country'].iloc[i] in [country for country in pays_with_euro.keys() if pays_with_euro[country] == 1]:
        euro_zone_rows.append(original_index)

len(euro_zone_rows)

---------------------------------------------------------------------------------------------------------------------------------------
THE CODE BELOW WAS USED TO MERGE THE COST CRITERIA DICTIONARY WITH THE EXISTING DATAFRAME TO COMPLEMENT THE TOTAL DATASET.
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
with open("Data/data_dict_2020_ac_cost.pickle", "rb") as f:
    data_dict_2020_ac_cost = pickle.load(f)

#transform dict of dicts to list of dicts
list_of_dicts = []
for key in data_dict_2020_ac_cost.keys():
    list_of_dicts.append(data_dict_2020_ac_cost[key])

#some rows apparenly have a crazy stupid length of over 2968 criteria... makes sense. TRIM THEM!!
key_value_remove_list = []

for row in list_of_dicts:
    for key in row.keys():
        col = int(key.split(":")[1])
        if col > 30:
            key_value_remove_list.append(key)

key_value_remove_list = set(key_value_remove_list)

for row in tqdm(list_of_dicts):
    for key in key_value_remove_list:
        if key in row.keys():
            del row[key]

#with open("Data/data_dict_2020_ac_cost_trimmed", "wb") as f:
#    pickle.dump(list_of_dicts, f)


#make a df from the data_dict_2020_ac_cost
df_ac_cost_2020 = pd.DataFrame.from_records(list_of_dicts)

for column in df_ac_cost_2020.columns:
    df_data_2020[column] = df_ac_cost_2020[column]

the one below is most recent 27-10

---------------------------------------------------------------------------------------------------------------------------------------
VECTORIZATION OF UNSTRUCTURED DATA
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

In [ ]:
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [ ]:
tokens = tokenizer(test_string_nl, return_tensors="pt", padding = True, truncation=True)
outputs = model(**tokens)
embedding = outputs.last_hidden_state

tokens_2 = tokenizer(test_string_nl_2, return_tensors="pt", padding = True, truncation=True)
outputs_2 = model(**tokens_2)
embedding_2 = outputs_2.last_hidden_state

print(embedding.shape, embedding_2.shape)

In [ ]:
test_string = ["Deze string bespreekt sporten, bewegen, gezond eten","Aarde bevat veel verschillende grondstoffen"]
tokens = tokenizer(test_string, return_tensors="pt", padding = True, truncation=True)
outputs = model(**tokens)
embedding = outputs.last_hidden_state

In [ ]:
print(embedding[0].shape, embedding[1].shape)

In [ ]:
cosi = torch.nn.CosineSimilarity(dim=1) 
output = cosi(embedding[0], embedding[1])
output

In [ ]:
tokens_2 = tokenizer(test_string_nl_2, return_tensors="pt", padding = True, truncation=True)
outputs_2 = model(**tokens_2)
embedding_2 = outputs_2.last_hidden_state

In [ ]:
cosi = torch.nn.CosineSimilarity() 
output = cosi(embedding, embedding_2)

In [ ]:
test_string_nl = "Deze string bespreekt sporten, bewegen, gezond eten"
#test_string_en = "The requested services mainly concern advising and negotiating the acquisition (particularly on an expropriation basis) and sale of immovable property with all associated preparatory work. Prior valuation of real estate objects is not part of this tender."
test_string_en = "This is a completely different string in which I will talk about airplanes, bikes, food and a bunch of random things"
encoded_input_nl = tokenizer.encode(test_string_nl, return_tensors="pt", padding = True, truncation = True)
encoded_input_en = tokenizer.encode(test_string_en, return_tensors="pt", padding = True, truncation = True)

with torch.no_grad():
    outputs_nl = model(encoded_input_en)
embedding_nl = outputs_nl.last_hidden_state

with torch.no_grad():
    outputs_en = model(encoded_input_en)
embedding_en = outputs_en.last_hidden_state

In [ ]:
embedding_nl.shape

In [ ]:
cosi = torch.nn.CosineSimilarity(dim=0) 
output = cosi(embedding_nl, embedding_en)
output

In [ ]:
model = SentenceTransformer('distiluse-base-multilingual-cased-v2')
sentences = ["Deze string bespreekt sporten, bewegen, gezond eten", "This string talks about sports, being active and heathy eating"]
sentences_2 = ["Deze string gaat over sporten, bewegen en gezond eten", "Aarde bevat veel verschillende grondstoffen"]
embeddings = model.encode(sentences, convert_to_tensor = True)

In [ ]:
cosi = torch.nn.CosineSimilarity(dim=0) 
output = cosi(embeddings[0], embeddings[1])
output


CODE FOR IMPUTING WITH MEAN BELOW

In [ ]:
#what was the number of bids when PT_AWARD_CONTRACT_WITHOUT_CALL? apparently empty which is strange. Offer is very often there so take the mean
total = 0
counter = 0
for i, value in enumerate(df["nb_tenders_received"].loc[df["procedure"] == "PT_AWARD_CONTRACT_WITHOUT_CALL"]):
    if np.isnan(value) == False:
        counter += 1
        total += value

mean_bids_no_cal = total / counter
mean_bids_no_cal

#what was the number of bids when PT_AWARD_CONTRACT_WITHOUT_CALL? apparently empty which is strange. Offer is very often there so take the mean
total = 0
counter = 0
for i, value in enumerate(df["nb_tenders_received"].loc[df["framework or dynamic purchasing"] != "no framework or dynamic purchasing"]):
    if np.isnan(value) == False:
        counter += 1
        total += value

mean_bids_dps = total / counter

#conclusion stands that we need to impute, but first lets change based on the findings we got from doing the research
#in case of pt_award_contract_no_call, we will impute the average. For the purchasing agreement we will use a bid of 1
for i in range(0, len(df)):
    if df["procedure"].iloc[i] == "PT_AWARD_CONTRACT_WITHOUT_CALL" and pd.isna(df["nb_tenders_received"].iloc[i]) == True:
        df.at[i, "nb_tenders_received"] = mean_bids_no_cal
    elif df["framework or dynamic purchasing"].iloc[i] != "no framework or dynamic purchasing" and pd.isna(df["nb_tenders_received"].iloc[i]) == True:
        df.at[i, "nb_tenders_received"] = mean_bids_dps
    else:
        continue

#all other cases we will just have to impute. lets replace with the mean
total = 0
counter = 0
for i in range(0, len(df)):
    if pd.isna(df["nb_tenders_received"].iloc[i]) != True:
        counter += 1
        total += df["nb_tenders_received"].iloc[i]

mean_bids = total / counter
mean_bids

for i in range(0, len(df)):
    if pd.isna(df["nb_tenders_received"].iloc[i]) == True:
        df.at[i, "nb_tenders_received"] = mean_bids

In [ ]:
#scale numerical columns for all datasets
scaler = StandardScaler()
for df_data in data_set_impute_split.keys():
    for col in numerical_columns:
        #reshape the data for each col in the train and test set
        data = train_test_data_dict[df_data][col].to_numpy().reshape(-1,1)
        train_test_data_dict[df_data][col] = scaler.fit_transform(data)

In [ ]:
#one-hot-encode the categorical features but not the target variable
#df_test = pd.get_dummies(df_test, columns = categorical_features)

#define target column
target_column = categorical_columns[9]
#initialize encoder
encoder = ce.BaseNEncoder(cols=base_n_encoder_cols, return_df=True, base=2)
#one-hot-encode categorical columns but remove the target_columns
one_hot_columns_no_target = [col for col in one_hot_columns if col != target_column]

data_set_impute_split["train_train"] = pd.get_dummies(data_set_impute_split["train_train"], columns = one_hot_columns_no_target, drop_first=True, dtype = int)
data_set_impute_split["train_test"] = pd.get_dummies(data_set_impute_split["train_test"], columns = one_hot_columns_no_target, drop_first=True, dtype = int)

#encode categorical columns with many unique values with the baseNencoder
data_set_impute_split["train_train"] = encoder.fit_transform(data_set_impute_split["train_train"])
data_set_impute_split["train_test"] = encoder.fit_transform(data_set_impute_split["train_test"])

In [ ]:
#handle difference in columns between train and test set
train_columns = data_set_impute_split["train_train"].columns
test_columns = data_set_impute_split["train_test"].columns
missing_columns = list(set(train_columns)- set(test_columns))

for col in missing_columns:
    if col not in train_columns:
        data_set_impute_split["train_train"][col] = 0
        print("column was added to the training set")
    elif col not in test_columns:
        data_set_impute_split["train_test"][col] = 0
        print("column was added to the test set")
    else:
        continue
    
print(missing_columns)

In [ ]:
#def create_mlp(dim, regress = False):
#let's build a sequential model for our numerical and categorical data
model = Sequential()
model.add(Dense(120, input_dim = 55, activation = "relu"))
model.add(Dense(60, activation = "relu"))
model.add(Dense(1, activation = "linear"))

model.compile(metrics = "mean_squared_error", loss = "mean_squared_error")

model.fit(X_train, y_train,
          batch_size = 30,
          epochs = 10,
          verbose = 1,
          validation_data = (X_test, y_test))

score = model.evaluate(X_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
#initialize tokenizer and model
model_name = "jplu/tf-xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)
xlm_roberta_model = TFXLMRobertaModel.from_pretrained(model_name)
max_seq_length = 124

tokenized_text = tokenizer(X_train_text_sample, return_tensors = "tf", padding=True, truncation=True)
input_ids = tokenized_text["input_ids"]
attention_mask = tokenized_text["attention_mask"]
#define inputs
input_layer = Input(shape = (max_seq_length,), name="input_ids")
attention_mask_layer = Input(shape=(max_seq_length,), name="attention_mask")

output = xlm_roberta_model(input_ids = input_ids, attention_mask = attention_mask)
model_after_embeddings = Model(inputs=[input_layer,attention_mask_layer], outputs=output.last_hidden_state)